In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
from agents import Agent, Runner, trace, Tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from openai import AsyncOpenAI



#### Model Setup

In [ ]:

client = AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=client
)

# client = AsyncOpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama"
# )

# model = OpenAIChatCompletionsModel(
#     model="qwen2.5:14b",
#     openai_client=client
# )

#### Flight and Hotel MCP as well as MCP memory

In [3]:
#!uv run ../tools/flight_operations_tools.py

In [4]:
flight_mcp = {"command": "uv", "args": ["run", "../tools/flight_operations_tools.py"]}
async with MCPServerStdio(params=flight_mcp, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [5]:
hotel_mcp = {"command": "uv", "args": ["run", "../tools/search_hotels.py"]}
async with MCPServerStdio(params=hotel_mcp, client_session_timeout_seconds=30) as server: 
    mcp_tools = await server.list_tools()


In [6]:
iata_lookup = {"command": "uv", "args": ["run", "../tools/search_airport.py"]}
async with MCPServerStdio(params=iata_lookup, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

In [7]:
# db_mcp = {"command": "npx", "args": ["-y", "mcp-memory-libsql"], "env": {"LIBSQL_URL": "file:memory/iata_code_lookup.db"}}
# async with MCPServerStdio(params=db_mcp, client_session_timeout_seconds=30) as server:
#     mcp_tools = await server.list_tools()

In [8]:
push_notif_mcp = {"command": "uv", "args": ["run", "../tools/push_server.py"]}
async with MCPServerStdio(params=push_notif_mcp, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

#### Configure multple MCPs (Flight and Hotel Operations + Push Notificatio)

In [9]:
travel_mcp_server_params = [ 
    flight_mcp, 
    hotel_mcp, 
    iata_lookup, 
    # db_mcp, 
    push_notif_mcp 
]

### Now create the MCPServerStdio for each

In [10]:
travel_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in travel_mcp_server_params]

In [11]:
async def travel_planner(travel_mcp_servers) -> Agent:
    instructions = """
                You are a travel planning agent.

                When airport information is needed:

                1. First use the airport lookup tool.
                2. If airport lookup returns airport details, use those details.
                3. Do not ask the user for airport details if they can be obtained from tools.

                When flight information is needed:

                1. Convert city names into IATA airport codes using airport lookup tools.
                2. Use flight tools only after airport codes are known.

                When hotel information is needed:

                1. Use hotel tools for destination city.

                When information is discovered that may be reused later:

                1. Search memory before repeating expensive lookups.
                2. Save useful lookup results into memory.

                Always prefer tool results over assumptions.

                Never invent airport codes, flight details, hotel details, prices, schedules, or availability.
                """
    planner = Agent(
        name="travel_planner",
        instructions=instructions,
        model=model,
        mcp_servers=travel_mcp_servers,
    )
    return planner

In [12]:
plan_travel = "Plan travel from Dubai to London, for 2 adults, starting from Dubai on 20-May, returning from london on 25-May, find the hotel, push notification with result"

for server in travel_mcp_servers:
    await server.connect()
researcher = await travel_planner(travel_mcp_servers)
with trace("travel_planner"):
    result = await Runner.run(researcher, plan_travel, max_turns=30)
display(Markdown(result.final_output))


It seems there was an error when trying to fetch the inventory information because a required parameter, `flight_id`, was missing. Could you please provide me with the `flight_id` so I can retrieve the correct details?